In [ ]:
import os
from dotenv import load_dotenv 
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
import bs4

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# data load
# https://towardsdatascience.com/
web_loader = WebBaseLoader("https://www.ufonies.com/blog.htm")

web_load = web_loader.load()
web_load

[Document(metadata={'source': 'https://www.ufonies.com/blog.htm', 'title': 'UFOnies Alien Blog UFO blog ufo humor alien humor Space Alien Blogs illustrated original content sci-fi blogs science fiction blog time travel historical fiction +UFOs ghosts paranormal humor blogs alien abduction humor ghosts humor blogs haunted places time travel humor space alien fun ufo alien encounters +humour +aliens +ufo', 'description': " Roz & Aileen's UFO Blog best ufo blog Most Popular UFO websites play the Magic Black Hole Answer Game original ufo Alien Art see what happens in an alien Abduction Room humour scifi original content science fiction blog historical fiction humor May Pang John Lennon's ufo sighting nyc the aliens view or version of john lennon and his near abduction by an alien +ufos photo of aliens fishing from dock paranormal ghost stories and photos astronomical illustrations syfy alien encounters alien photography Elvis Constellation Elvis Cartoons & Caricatures best sci-fi space tra

In [5]:
#data transform - text splitter - data to chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 50)
final_docs = text_splitter.split_documents(web_load)
final_docs[0]

Document(metadata={'source': 'https://www.ufonies.com/blog.htm', 'title': 'UFOnies Alien Blog UFO blog ufo humor alien humor Space Alien Blogs illustrated original content sci-fi blogs science fiction blog time travel historical fiction +UFOs ghosts paranormal humor blogs alien abduction humor ghosts humor blogs haunted places time travel humor space alien fun ufo alien encounters +humour +aliens +ufo', 'description': " Roz & Aileen's UFO Blog best ufo blog Most Popular UFO websites play the Magic Black Hole Answer Game original ufo Alien Art see what happens in an alien Abduction Room humour scifi original content science fiction blog historical fiction humor May Pang John Lennon's ufo sighting nyc the aliens view or version of john lennon and his near abduction by an alien +ufos photo of aliens fishing from dock paranormal ghost stories and photos astronomical illustrations syfy alien encounters alien photography Elvis Constellation Elvis Cartoons & Caricatures best sci-fi space trav

In [6]:
#Embeddings => text to vectors
# embeddings = OpenAIEmbeddings() => not paid 

embeddings = OllamaEmbeddings(model="nomic-embed-text")


C:\Users\HP\AppData\Local\Temp\ipykernel_876\1828889057.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [7]:
#vectorstore db
db = FAISS.from_documents(final_docs,embeddings)
db

In [8]:
#query from vectorStore db
query = "what happened on august 23 1974?"
result = db.similarity_search(query)
result[0].page_content

'actually considered “beaming him up” or abducting\n              him for a while, but I think they were more shocked then he was.\n              My mom did the above caricature from what\n              she remembers of that day. They made a big impression on him too\n                    I guess. His Walls and Bridges album has "On August 23 1974,\n                    I saw a UFO J.L." printed on the back. He also wrote; "There\'s\n                    UFO\'s over New York and I ain\'t too surprised." This was\n                    in his song "Nobody Told Me" probably written long before\n                    it was released on his Milk and Honey album, two years after\n                    his death. \nIf\n                    John was alive today my mom feels that he and Yoko would\n                    probably be involved in NYC politics.'

In [9]:
# load the model
#loading the llm model , since gpt is not free using ollama
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o")
# llm 
from langchain_ollama import ChatOllama

# llm = ChatOllama(model="llama3")
llm = ChatOllama(model="gemma:2b")
llm

ChatOllama(model='gemma:2b')

In [19]:
# creaating document chain

from langchain_core.prompts import ChatPromptTemplate
# from langchain_community.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain


prompt = ChatPromptTemplate.from_template(
"""
You are a helpful assistant.

Answer the question using the provided context.

If the answer is in the context, explain it clearly.
If the answer is partially in the context, use the context and summarize it.

Context:
{context}

Question:
{input}

Answer:
"""
)


document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant.\n\nAnswer the question using the provided context.\n\nIf the answer is in the context, explain it clearly.\nIf the answer is partially in the context, use the context and summarize it.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n'), additional_kwargs={})])
| ChatOllama(model='gemma:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [20]:
# Your chain expects:

# input   → question
# context → retrieved documents

# So the format should always be:

# Question + Context → Answer

from langchain_core.documents import Document
document_chain.invoke({
     "input":"who is mentioned in the story?",
    "context":[Document(page_content="It's getting near that time of the year again, Halloween! So I dusted off my Ouija board, lit some candles, got out my brushes, some watercolor paper and conjured up this. Recognize her, this was a tough little wicked witch of the southwest named Bonnie Parker")]
})

'The answer is Bonnie Parker.\n\nThe context mentions that the story is about a Halloween ritual involving a Ouija board and witches. Bonnie Parker is a character in the story who is mentioned as being a wicked witch of the southwest.'

In [21]:
db

In [22]:
#  creating retreival chain
from langchain_classic.chains import create_retrieval_chain

retriever = db.as_retriever()

retrieval_chain = create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000193C0ADD9D0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant.\n\nAnswer the question using the provided context.\n\nIf the answer is in the context, explain it cl

In [25]:
response = retrieval_chain.invoke(
    {"input": "Are there any paranormal stories reported?"}
)

print(response["answer"])

The context does not mention any paranormal stories reported, so I cannot answer this question from the provided context.


In [23]:
response = retrieval_chain.invoke(
    {"input": "who is bonnie parker"}
)

print(response["answer"])

Bonnie Parker is a wicked witch of the southwest named in the context. She traded in her broomstick for a Ford V-8, Clyde Barrow replaced the munchkins and the pumpkins were usually used for target practice.


In [15]:
#Debugging
docs = retriever.invoke("bonnie parker")

print(len(docs))

for d in docs:
    print(d.page_content)
    print("------")

4
actually considered “beaming him up” or abducting
              him for a while, but I think they were more shocked then he was.
              My mom did the above caricature from what
              she remembers of that day. They made a big impression on him too
                    I guess. His Walls and Bridges album has "On August 23 1974,
                    I saw a UFO J.L." printed on the back. He also wrote; "There's
                    UFO's over New York and I ain't too surprised." This was
                    in his song "Nobody Told Me" probably written long before
                    it was released on his Milk and Honey album, two years after
                    his death. 
If
                    John was alive today my mom feels that he and Yoko would
                    probably be involved in NYC politics.
------
Another movie I'd like to see would be a remake of Superman starring Arnold Schwarzenegger. Only instead of landing in Smallville as a baby he lands in Austr

In [16]:
print(len(final_docs))

71


In [17]:
print(final_docs[0].page_content)

UFOnies Alien Blog UFO blog ufo humor alien humor Space Alien Blogs illustrated original content sci-fi blogs science fiction blog time travel historical fiction +UFOs ghosts paranormal humor blogs alien abduction humor ghosts humor blogs haunted places time travel humor space alien fun ufo alien encounters +humour +aliens +ufo












CLICK This link to go to the MOBILE Version of FAN MAIL and other fun stuff.


 
 
 



CLICK above 
for my photos of Faces In Strange Places. 
If
          you have
          any questions, need any advice, need a laugh, play Aileen's Magic
          Black Hole Answer Game. Just click the icon below.

You
        can also get your caricature drawn online by an alien while you play.

CLICK This link for the MOBILE version to get a free caricature drawing of you online!


In [18]:
docs = retriever.invoke("bonnie parker")
print(docs)

[Document(id='342796b2-9a28-4527-8413-50445763409c', metadata={'source': 'https://www.ufonies.com/blog.htm', 'title': 'UFOnies Alien Blog UFO blog ufo humor alien humor Space Alien Blogs illustrated original content sci-fi blogs science fiction blog time travel historical fiction +UFOs ghosts paranormal humor blogs alien abduction humor ghosts humor blogs haunted places time travel humor space alien fun ufo alien encounters +humour +aliens +ufo', 'description': " Roz & Aileen's UFO Blog best ufo blog Most Popular UFO websites play the Magic Black Hole Answer Game original ufo Alien Art see what happens in an alien Abduction Room humour scifi original content science fiction blog historical fiction humor May Pang John Lennon's ufo sighting nyc the aliens view or version of john lennon and his near abduction by an alien +ufos photo of aliens fishing from dock paranormal ghost stories and photos astronomical illustrations syfy alien encounters alien photography Elvis Constellation Elvis C